# Preprocessing

Per-sample QC filtering, then concatenation and normalization.
Set `DATA` in `config.py` before running.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad

from config import DATA, REPO

## 1. Load the sample manifest

`sample_metadata.csv` maps each file to its condition and age. To add or remove a sample, edit the manifest, not this notebook.

In [ ]:
meta = pd.read_csv(REPO / 'sample_metadata.csv').set_index('file')
metacols = list(meta.columns)  # type, sample, age
meta

## 2. Per-sample QC filtering

Standard QC thresholds for mouse single-cell data:

- `min_genes=200`, `min_cells=3`
- gene counts trimmed to the 2nd-99th percentile **per sample**, so the cut
  adapts to each sample's sequencing depth rather than imposing one global limit
- `pct_counts_mt < 10`
- `pct_counts_ribo < 20`

In [ ]:
adatas = []

for fname, row in meta.iterrows():
    path = DATA / fname
    if not path.exists():
        raise FileNotFoundError(
            f"{fname} is listed in sample_metadata.csv but was not found in {DATA}"
        )

    adata = sc.read_h5ad(path)
    adata.var_names_make_unique()

    # attach condition / age metadata to every cell from this sample
    for c in metacols:
        adata.obs[c] = row[c]

    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)

    adata.var['mt']   = adata.var_names.str.startswith('mt-')
    adata.var['ribo'] = adata.var_names.str.startswith(('Rpl', 'Rps'))

    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt', 'ribo'], percent_top=None,
        log1p=False, inplace=True
    )

    upperlim = np.percentile(adata.obs['n_genes_by_counts'], 99)
    lowerlim = np.percentile(adata.obs['n_genes_by_counts'], 2)

    adata = adata[(adata.obs['n_genes_by_counts'] >= lowerlim)
                  & (adata.obs['n_genes_by_counts'] <= upperlim)
                  & (adata.obs['pct_counts_mt'] < 10)
                  & (adata.obs['pct_counts_ribo'] < 20)
                  ].copy()

    # 10x barcodes repeat across samples - suffix them so each cell stays
    # traceable to its sample after concatenation
    adata.obs_names = adata.obs_names.astype(str) + '_' + str(row['sample'])

    adatas.append(adata)
    print(adata.shape, fname)

## 3. Concatenate

In [ ]:
adata = ad.concat(adatas, join='outer')  # keep the union of genes across samples
adata.obs_names_make_unique()            # backstop in case a sample name repeats
adata

## 4. Normalize

Raw counts are copied into `.layers['counts']` **before** normalizing. scVI
(the next step) trains on integer counts, so this layer is required downstream.

In [ ]:
adata.layers['counts'] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## 5. QC check

Look at these before moving on. If one sample looks unlike the others *after*
filtering, that is worth knowing before it propagates through integration.

In [ ]:
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'pct_counts_mt', 'pct_counts_ribo'],
    groupby='sample', rotation=90, jitter=0.4
)

## 6. Save

In [ ]:
adata.write_h5ad(DATA / 'nerandomilast_preprocessed.h5ad')